In [ ]:
import scanpy as sc
import pandas as pd

In [ ]:
adata = sc.read_h5ad("raw/ReplogleWeissman2022_K562_essential.h5ad")

In [ ]:
adata

In [ ]:
adata.layers["counts"] = adata.X.copy()

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [ ]:
sc.pp.highly_variable_genes(adata,n_top_genes=2000)
adata = adata[:, adata.var['highly_variable']]

In [ ]:
adata.write_h5ad(f"raw/Replogle_hvg2000.h5ad")

In [ ]:
adata = sc.read_h5ad("raw/Replogle_hvg2000.h5ad")

In [ ]:
adata

In [ ]:
adata.obs["perturbation"].value_counts()

In [ ]:
adata.obs["perturbation"]

In [ ]:
df = pd.DataFrame(
    adata.obs["perturbation"].unique(),
    columns=["gene"]
)

df.to_csv("target_genes_replogle_essential.csv", index=False)

In [ ]:
adata.obs["perturbation"].value_counts()

In [ ]:
df_sub = pd.read_csv('./annotated_embedding_coordinates.csv')
selected_genes = list(df_sub['gene'].values) + ['non-targeting']

In [ ]:
adata_big = sc.read_h5ad("raw/ReplogleWeissman2022_K562_gwps.h5ad")
adata_big = adata_big[adata_big.obs['gene'].isin(selected_genes), :].copy()
sc.pp.normalize_total(adata_big, target_sum=1e4)
sc.pp.log1p(adata_big)

In [ ]:
adata = sc.read_h5ad("raw/ReplogleWeissman2022_K562_essential.h5ad")
adata = adata[adata.obs['gene'].isin(selected_genes), :].copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [ ]:
adata

In [ ]:
adata_big

In [ ]:
# 1) 准备成 systema 对齐风格
adata_sys = prepare_adata_for_systema_cc(
    adata,
    n_top_genes=5000,
    normalize_if_needed=False,   # 你当前对象建议 False
    use_symbol_update=False,     # 更贴近 systema 原版
    subset_hvg_and_targets=True, # 更贴近 systema
    copy=True,
)

# 2) 跑 cell cycle
adata_cc, cc_info = run_systema_aligned_cell_cycle(
    adata_sys,
    cell_cycle_gene_file="./regev_lab_cell_cycle_genes.txt",
    use_symbol_update=False,     # 先别开，先对齐 systema
)

# 3) 画图
fig, ax, summary = plot_phase_density(
    adata_cc,
    control_col="control",
    phase_col="phase",
    title="Phase Density",
    figsize=(6, 7),
    fontsize=16,
    ylim=(0, 0.6),
    save="replogle_essential_cell_cycle.pdf",
)

plt.show()


In [ ]:
# 1) 准备成 systema 对齐风格
adata_sys = prepare_adata_for_systema_cc(
    adata_big,
    n_top_genes=5000,
    normalize_if_needed=False,   # 你当前对象建议 False
    use_symbol_update=False,     # 更贴近 systema 原版
    subset_hvg_and_targets=True, # 更贴近 systema
    copy=True,
)

# 2) 跑 cell cycle
adata_cc, cc_info = run_systema_aligned_cell_cycle(
    adata_sys,
    cell_cycle_gene_file="./regev_lab_cell_cycle_genes.txt",
    use_symbol_update=False,     # 先别开，先对齐 systema
)

# 3) 画图
fig, ax, summary = plot_phase_density(
    adata_cc,
    control_col="control",
    phase_col="phase",
    title="Phase Density",
    figsize=(6, 7),
    fontsize=16,
    ylim=(0, 0.6),
    save="replogle_gwps_cell_cycle.pdf",
)

plt.show()


In [ ]:
# 1. 看看 CSV 里到底有什么列和内容
df_sub = pd.read_csv('./annotated_embedding_coordinates.csv')
print(df_sub.columns.tolist())
print(df_sub.shape)
print(df_sub.head())

# 2. 看看是否有分组/类别列
for col in df_sub.columns:
    if df_sub[col].nunique() < 30:
        print(f"\n--- {col} ---")
        print(df_sub[col].value_counts())

# 3. 看看你当前 adata 的 control 实际情况
print("\n--- control mask ---")
print(adata_cc.obs['control'].value_counts())
print(adata_cc.obs['control_str'].value_counts())

# 4. 看看 phase 在两组中的实际分布
c_mask = adata_cc.obs['control'].astype(bool)
print("\n--- Control phase ---")
print(adata_cc.obs.loc[c_mask, 'phase'].value_counts(normalize=True))
print("\n--- Perturbed phase ---")
print(adata_cc.obs.loc[~c_mask, 'phase'].value_counts(normalize=True))

# 5. 关键：看看是不是某些特定 perturbation 才有剧烈变化
phase_by_gene = adata_cc.obs.groupby('gene')['phase'].value_counts(normalize=True).unstack(fill_value=0)
# 看哪些 perturbation 的 G2M 或 S 比例和 control 差异最大
ctrl_dist = adata_cc.obs.loc[c_mask, 'phase'].value_counts(normalize=True)
phase_by_gene['G2M_diff'] = (phase_by_gene.get('G2M', 0) - ctrl_dist.get('G2M', 0)).abs()
print("\n--- Top 20 perturbations by G2M shift ---")
print(phase_by_gene.sort_values('G2M_diff', ascending=False).head(20))


In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import anndata
import seaborn as sns
import matplotlib.pyplot as plt

from scipy.spatial.distance import jensenshannon
from scipy.stats import chi2_contingency


# -----------------------------
# 1) 工具函数：判断当前基因名列
# -----------------------------
def infer_gene_symbols(adata):
    """
    返回一个可用于 cell-cycle gene matching 的 gene symbol 列表。
    优先级：
    1. var['gene_name']
    2. var['gene_symbol']
    3. var index / var_names
    """
    for col in ["gene_name", "gene_symbol", "symbol", "gene"]:
        if col in adata.var.columns:
            vals = adata.var[col].astype(str).values
            if pd.Series(vals).nunique() > 100:
                return pd.Index(vals), col

    return pd.Index(adata.var_names.astype(str)), "var_names"


# -----------------------------
# 2) 工具函数：读入 systema / Regev cell-cycle 基因列表
# -----------------------------
def load_regev_cell_cycle_genes(cell_cycle_gene_file):
    """
    systema 风格：前43个是 S，后面是 G2M
    """
    genes = [x.strip() for x in open(cell_cycle_gene_file) if x.strip()]
    s_genes = genes[:43]
    g2m_genes = genes[43:]
    return s_genes, g2m_genes


# -----------------------------
# 3) 工具函数：可选 symbol update（默认关闭，因 systema 原版没开）
# -----------------------------
def remap_gene_list(gene_list, use_symbol_update=False):
    """
    use_symbol_update=False 更接近 systema 原始实现
    use_symbol_update=True 更接近 Seurat updated symbols
    """
    if not use_symbol_update:
        return list(dict.fromkeys(gene_list))

    symbol_update_map = {
        "MLF1IP": "CENPU",
        "FAM64A": "PIMREG",
        "HN1": "JPT1",
        "MCM2": "MCM7",
        "RPA2": "POLR1B",
        "BRIP1": "MRPL36",
    }
    return list(dict.fromkeys([symbol_update_map.get(g, g) for g in gene_list]))


# -----------------------------
# 4) 核心：构建 systema 对齐版 adata
# -----------------------------
def prepare_adata_for_systema_cc(
    adata,
    n_top_genes=5000,
    normalize_if_needed=False,
    target_sum=1e4,
    use_symbol_update=False,
    subset_hvg_and_targets=True,
    copy=True,
):
    """
    尽量对齐 systema 的思路：
    - 可选 normalize_total + log1p
    - 选择用于匹配的 gene symbols
    - 计算 HVG
    - 保留 HVG + perturbation target genes
    - 设置 var_names 为 gene symbols，供 score_genes_cell_cycle 使用

    参数说明：
    normalize_if_needed:
        False: 默认不重复 normalize/log1p（适合你这个已有 uns['log1p'] 的对象）
        True: 如果你确认当前 adata 还没标准化/没log1p，再打开
    """
    ad = adata.copy() if copy else adata

    # 1) 预处理：只在需要时做
    already_logged = "log1p" in ad.uns
    if normalize_if_needed:
        sc.pp.normalize_total(ad, target_sum=target_sum)
        sc.pp.log1p(ad)
    else:
        if not already_logged:
            print("[Warning] adata.uns 中没有 'log1p'，但你设置了 normalize_if_needed=False。")
            print("          如果当前矩阵还没 log1p，建议改成 normalize_if_needed=True。")

    # 2) 推断 gene symbols
    gene_symbols, gene_symbol_source = infer_gene_symbols(ad)

    # 3) 写入一个统一列，方便后面使用
    ad.var["__gene_symbol__"] = gene_symbols.astype(str)

    # 4) 设置 var_names 为 gene symbols，和 systema 思路一致
    ad.var_names = ad.var["__gene_symbol__"].astype(str)
    ad.var_names_make_unique()

    # 5) HVG
    sc.pp.highly_variable_genes(ad, n_top_genes=n_top_genes, subset=False)

    # 6) systema 风格：HVG + perturbation target genes
    # 你的 obs 里有 gene，可直接用
    if subset_hvg_and_targets:
        hvg_flag = ad.var["highly_variable"].values

        if "gene" in ad.obs.columns:
            target_genes = pd.Index(ad.obs["gene"].astype(str).unique())
            target_flag = ad.var["__gene_symbol__"].isin(target_genes).values
        else:
            target_flag = np.zeros(ad.n_vars, dtype=bool)

        select_flag = hvg_flag | target_flag
        ad = ad[:, select_flag].copy()

    # 7) control 标志，尽量兼容你的字段
    if "control" not in ad.obs.columns:
        if "gene" in ad.obs.columns:
            ad.obs["control"] = ad.obs["gene"].astype(str).isin(["non-targeting", "NT", "control"])
        else:
            ad.obs["control"] = False

    ad.obs["control"] = ad.obs["control"].astype(bool)
    ad.obs["control_str"] = np.where(ad.obs["control"], "Control", "Perturbed")

    print("=== prepare_adata_for_systema_cc ===")
    print(f"Gene symbol source: {gene_symbol_source}")
    print(f"Shape after prepare: {ad.shape}")
    print(f"Already logged: {already_logged}")
    print(f"Control cells: {ad.obs['control'].sum()} / {ad.n_obs}")

    return ad


# -----------------------------
# 5) 打分：systema 对齐版 cell-cycle scoring
# -----------------------------
def run_systema_aligned_cell_cycle(
    adata,
    cell_cycle_gene_file,
    use_symbol_update=False,
):
    """
    在 prepare 后的 adata 上运行 cell-cycle scoring
    """
    ad = adata.copy()

    s_genes_raw, g2m_genes_raw = load_regev_cell_cycle_genes(cell_cycle_gene_file)

    s_genes = remap_gene_list(s_genes_raw, use_symbol_update=use_symbol_update)
    g2m_genes = remap_gene_list(g2m_genes_raw, use_symbol_update=use_symbol_update)

    s_genes_use = [g for g in s_genes if g in ad.var_names]
    g2m_genes_use = [g for g in g2m_genes if g in ad.var_names]

    print("=== run_systema_aligned_cell_cycle ===")
    print(f"Matched S genes: {len(s_genes_use)}")
    print(f"Matched G2M genes: {len(g2m_genes_use)}")
    print("Some matched S genes:", s_genes_use[:10])
    print("Some matched G2M genes:", g2m_genes_use[:10])

    if len(s_genes_use) < 10 or len(g2m_genes_use) < 10:
        print("[Warning] matched genes 太少，结果可能不稳定。请重点检查 gene symbol 来源。")

    sc.tl.score_genes_cell_cycle(
        ad,
        s_genes=s_genes_use,
        g2m_genes=g2m_genes_use,
    )

    print(ad.obs["phase"].value_counts())
    print(ad.obs["phase"].value_counts(normalize=True))

    return ad, {
        "s_genes_use": s_genes_use,
        "g2m_genes_use": g2m_genes_use,
    }


# -----------------------------
# 6) 绘图：复现 systema 风格的 phase density barplot
# -----------------------------
def plot_phase_density(
    adata_cc,
    control_col="control",
    phase_col="phase",
    title="Phase Density",
    figsize=(6, 7),
    fontsize=16,
    ylim=(0, 0.6),
    save=None,
):
    """
    画 Control vs Perturbed 的 phase 分布柱状图
    返回: fig, ax, summary_dict
    """
    ad = adata_cc.copy()

    if control_col not in ad.obs.columns:
        raise ValueError(f"{control_col} 不在 adata.obs 中")
    if phase_col not in ad.obs.columns:
        raise ValueError(f"{phase_col} 不在 adata.obs 中")

    c_mask = ad.obs[control_col].astype(bool)

    c = ad.obs.loc[c_mask, phase_col].value_counts()
    pc = ad.obs.loc[~c_mask, phase_col].value_counts()

    # 固定相位顺序，避免漏类时报错
    phase_order = ["G1", "S", "G2M"]
    c = c.reindex(phase_order, fill_value=0)
    pc = pc.reindex(phase_order, fill_value=0)

    c_norm = c / c.sum() if c.sum() > 0 else c.astype(float)
    pc_norm = pc / pc.sum() if pc.sum() > 0 else pc.astype(float)

    df = pd.DataFrame({
        "Condition": ["Control"] * 3 + ["Perturbed"] * 3,
        "Phase": phase_order * 2,
        "Density": list(c_norm.values) + list(pc_norm.values),
    })

    sns.set_style("whitegrid")
    fig, ax = plt.subplots(figsize=figsize)

    palette = {
        "G1": "#C8DBC8",
        "S": "#8DB081",
        "G2M": "#3E4A3B",
    }

    sns.barplot(
        data=df,
        x="Condition",
        y="Density",
        hue="Phase",
        hue_order=phase_order,
        palette=palette,
        ax=ax,
    )

    # 数值标注
    for bar in ax.patches:
        height = bar.get_height()
        if np.isfinite(height) and height > 0:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                height + 0.01,
                f"{height:.2f}",
                ha="center",
                va="bottom",
                fontsize=fontsize - 2,
            )

    # 卡方检验
    table = np.array([c.values, pc.values])
    chi2, p, _, _ = chi2_contingency(table)

    # JS distance
    js = jensenshannon(c_norm.values, pc_norm.values)

    # 显著性标注
    y_max = df["Density"].max()
    y_line = min(y_max * 1.12 + 0.02, (ylim[1] - 0.03) if ylim is not None else y_max + 0.08)
    x1, x2 = 0, 1

    ax.plot(
        [x1, x1, x2, x2],
        [y_line - 0.01, y_line, y_line, y_line - 0.01],
        lw=1,
        color="black",
    )

    p_text = f"χ²={chi2:.2f}, p={p:.1e}, JS={js:.2f}"
    ax.text(
        (x1 + x2) / 2,
        y_line + 0.015,
        p_text,
        ha="center",
        va="bottom",
        fontsize=fontsize - 2,
    )

    ax.set_title(title, fontsize=fontsize)
    ax.set_ylabel("Density", fontsize=fontsize)
    ax.set_xlabel("")
    ax.tick_params(axis="x", labelsize=fontsize)
    ax.tick_params(axis="y", labelsize=fontsize)

    legend = ax.legend(
        title="",
        fontsize=fontsize - 2,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.08),
        fancybox=True,
        shadow=False,
        ncol=3,
    )

    ax.set_axisbelow(True)
    ax.grid(linestyle="dotted", axis="y")

    if ylim is not None:
        ax.set_ylim(ylim)

    plt.tight_layout()

    if save is not None:
        plt.savefig(save, bbox_inches="tight")

    summary = {
        "control_counts": c,
        "perturbed_counts": pc,
        "control_density": c_norm,
        "perturbed_density": pc_norm,
        "chi2": chi2,
        "pvalue": p,
        "js_distance": js,
        "plot_df": df,
    }

    return fig, ax, summary


In [ ]:
import anndata
import scanpy as sc
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.spatial.distance import jensenshannon
from scipy.stats import chi2_contingency

# ========== 1. 读原始数据 ==========
adata_raw = anndata.read_h5ad("raw/ReplogleWeissman2022_K562_essential.h5ad")
print("原始shape:", adata_raw.shape)
print("var_names[:10]:", adata_raw.var_names[:10].tolist())
print("var columns:", adata_raw.var.columns.tolist())

# ========== 2. CSV 筛选 ==========
df_sub = pd.read_csv('./annotated_embedding_coordinates.csv')
selected_genes = list(df_sub['gene'].values) + ['non-targeting']
adata_ = adata_raw[adata_raw.obs['gene'].isin(selected_genes), :].copy()
print("CSV筛选后:", adata_.shape)

# ========== 3. Normalize + log1p（从 raw 开始，只做一次）==========
sc.pp.normalize_total(adata_)
sc.pp.log1p(adata_)

# ========== 4. HVG ==========
sc.pp.highly_variable_genes(adata_, n_top_genes=5000, subset=False)

# ========== 5. 确定 gene symbol 列 ==========
# 根据你的 var 结构适配
if 'gene_name' in adata_.var.columns:
    gene_col = 'gene_name'
else:
    # var_names 本身就是 gene symbol
    adata_.var['gene_name'] = adata_.var_names.astype(str)
    gene_col = 'gene_name'

# ========== 6. select_flag: HVG + target genes ==========
hvg_flag = adata_.var['highly_variable'].values
gene_flag = adata_.var[gene_col].isin(adata_.obs['gene'].values).values
select_flag = hvg_flag | gene_flag

# ========== 7. condition_flag: perturbation target 在 var 中 ==========
condition_flag = adata_.obs['gene'].isin(
    adata_.var[gene_col].values.tolist() + ['non-targeting']
).values

print(f"condition_flag 保留细胞: {condition_flag.sum()} / {adata_.n_obs}")

# ========== 8. 最终子集 ==========
adata_subset = adata_[condition_flag, :][:, select_flag].copy()
adata_subset.obs['control'] = adata_subset.obs['gene'] == 'non-targeting'
print("最终subset shape:", adata_subset.shape)

# ==========================================================
# 9. Cell Cycle Scoring —— 关键：在全基因集上打分！！
# ==========================================================
# 用 condition_flag 筛选后的全基因版本，不是 HVG 子集
adata_cc = adata_[condition_flag, :].copy()
adata_cc.var_names = adata_cc.var[gene_col].astype(str)
adata_cc.var_names_make_unique()

adata_cc.obs['control'] = adata_cc.obs['gene'] == 'non-targeting'

cell_cycle_genes = [x.strip() for x in open('./regev_lab_cell_cycle_genes.txt')]
s_genes = cell_cycle_genes[:43]
g2m_genes = cell_cycle_genes[43:]

s_use = [g for g in s_genes if g in adata_cc.var_names]
g2m_use = [g for g in g2m_genes if g in adata_cc.var_names]
print(f"Matched S: {len(s_use)}/{len(s_genes)}, G2M: {len(g2m_use)}/{len(g2m_genes)}")

sc.tl.score_genes_cell_cycle(adata_cc, s_genes=s_use, g2m_genes=g2m_use)

# ========== 10. 检查分布 ==========
c_mask = adata_cc.obs['control'].astype(bool)
print("\n--- Control ---")
print(adata_cc.obs.loc[c_mask, 'phase'].value_counts(normalize=True))
print("\n--- Perturbed ---")
print(adata_cc.obs.loc[~c_mask, 'phase'].value_counts(normalize=True))

# ========== 11. 画图（严格按 systema 原版）==========
data = []
c = adata_cc[c_mask].obs['phase'].value_counts()
pc = adata_cc[~c_mask].obs['phase'].value_counts()
categories = ['G1', 'S', 'G2M']

c = c.reindex(categories, fill_value=0)
pc = pc.reindex(categories, fill_value=0)
c_norm = c / c.sum()
pc_norm = pc / pc.sum()

for phase in categories:
    data.append(['Control', phase, c_norm[phase]])
    data.append(['Perturbed', phase, pc_norm[phase]])

df = pd.DataFrame(data, columns=['Condition', 'Phase', 'Density'])

sns.set_style('whitegrid')
fontsize = 20

plt.figure(figsize=(6, 7))
ax = sns.barplot(data=df, x='Condition', y='Density', hue='Phase',
                 palette={"S": "#8DB081", "G1": "#C8DBC8", "G2M": "#3E4A3B"},
                 hue_order=['G1', 'S', 'G2M'])

for bar in ax.patches:
    height = bar.get_height()
    if height > 0:
        ax.text(bar.get_x() + bar.get_width()/2, height + 0.01,
                f'{height:.2f}', ha='center', fontsize=fontsize-2)

table = np.array([c.values, pc.values])
chi2, p, _, _ = chi2_contingency(table)
js = jensenshannon(c_norm.values, pc_norm.values)

y_max = df['Density'].max()
y_line = y_max * 1.1
ax.plot([0, 0, 1, 1], [y_line-0.01, y_line, y_line, y_line-0.01], lw=1, color='black')
p_text = f"χ²={chi2:.2f}, p={p:.1e}, JS={js:.2f}"
ax.text(0.5, y_line + 0.02, p_text, ha='center', fontsize=fontsize-2)

ax.set_title('Phase Density', fontsize=fontsize)
ax.set_ylabel('Density', fontsize=fontsize)
ax.set_xlabel('')
plt.xticks(fontsize=fontsize)
plt.yticks(fontsize=fontsize)
plt.legend(title='', fontsize=fontsize-2, loc='upper center',
           bbox_to_anchor=(0.5, -0.1), fancybox=True, shadow=False, ncol=3)
plt.gca().set_axisbelow(True)
plt.ylim((0, 0.6))
plt.tight_layout()
plt.grid(linestyle='dotted', axis='y')
plt.savefig('systema_reproduced.pdf', bbox_inches='tight')
plt.show()


In [ ]:
import anndata
import scanpy as sc
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.spatial.distance import jensenshannon
from scipy.stats import chi2_contingency

# ========== 1. 读原始数据 ==========
adata_raw = anndata.read_h5ad("raw/ReplogleWeissman2022_K562_gwps.h5ad")
print("原始shape:", adata_raw.shape)
print("var_names[:10]:", adata_raw.var_names[:10].tolist())
print("var columns:", adata_raw.var.columns.tolist())

# ========== 2. CSV 筛选 ==========
df_sub = pd.read_csv('./annotated_embedding_coordinates.csv')
selected_genes = list(df_sub['gene'].values) + ['non-targeting']
adata_ = adata_raw[adata_raw.obs['gene'].isin(selected_genes), :].copy()
print("CSV筛选后:", adata_.shape)

# ========== 3. Normalize + log1p（从 raw 开始，只做一次）==========
sc.pp.normalize_total(adata_)
sc.pp.log1p(adata_)

# ========== 4. HVG ==========
sc.pp.highly_variable_genes(adata_, n_top_genes=5000, subset=False)

# ========== 5. 确定 gene symbol 列 ==========
# 根据你的 var 结构适配
if 'gene_name' in adata_.var.columns:
    gene_col = 'gene_name'
else:
    # var_names 本身就是 gene symbol
    adata_.var['gene_name'] = adata_.var_names.astype(str)
    gene_col = 'gene_name'

# ========== 6. select_flag: HVG + target genes ==========
hvg_flag = adata_.var['highly_variable'].values
gene_flag = adata_.var[gene_col].isin(adata_.obs['gene'].values).values
select_flag = hvg_flag | gene_flag

# ========== 7. condition_flag: perturbation target 在 var 中 ==========
condition_flag = adata_.obs['gene'].isin(
    adata_.var[gene_col].values.tolist() + ['non-targeting']
).values

print(f"condition_flag 保留细胞: {condition_flag.sum()} / {adata_.n_obs}")

# ========== 8. 最终子集 ==========
adata_subset = adata_[condition_flag, :][:, select_flag].copy()
adata_subset.obs['control'] = adata_subset.obs['gene'] == 'non-targeting'
print("最终subset shape:", adata_subset.shape)

# ==========================================================
# 9. Cell Cycle Scoring —— 关键：在全基因集上打分！！
# ==========================================================
# 用 condition_flag 筛选后的全基因版本，不是 HVG 子集
adata_cc = adata_[condition_flag, :].copy()
adata_cc.var_names = adata_cc.var[gene_col].astype(str)
adata_cc.var_names_make_unique()

adata_cc.obs['control'] = adata_cc.obs['gene'] == 'non-targeting'

cell_cycle_genes = [x.strip() for x in open('./regev_lab_cell_cycle_genes.txt')]
s_genes = cell_cycle_genes[:43]
g2m_genes = cell_cycle_genes[43:]

s_use = [g for g in s_genes if g in adata_cc.var_names]
g2m_use = [g for g in g2m_genes if g in adata_cc.var_names]
print(f"Matched S: {len(s_use)}/{len(s_genes)}, G2M: {len(g2m_use)}/{len(g2m_genes)}")

sc.tl.score_genes_cell_cycle(adata_cc, s_genes=s_use, g2m_genes=g2m_use)

# ========== 10. 检查分布 ==========
c_mask = adata_cc.obs['control'].astype(bool)
print("\n--- Control ---")
print(adata_cc.obs.loc[c_mask, 'phase'].value_counts(normalize=True))
print("\n--- Perturbed ---")
print(adata_cc.obs.loc[~c_mask, 'phase'].value_counts(normalize=True))

# ========== 11. 画图（严格按 systema 原版）==========
data = []
c = adata_cc[c_mask].obs['phase'].value_counts()
pc = adata_cc[~c_mask].obs['phase'].value_counts()
categories = ['G1', 'S', 'G2M']

c = c.reindex(categories, fill_value=0)
pc = pc.reindex(categories, fill_value=0)
c_norm = c / c.sum()
pc_norm = pc / pc.sum()

for phase in categories:
    data.append(['Control', phase, c_norm[phase]])
    data.append(['Perturbed', phase, pc_norm[phase]])

df = pd.DataFrame(data, columns=['Condition', 'Phase', 'Density'])

sns.set_style('whitegrid')
fontsize = 20

plt.figure(figsize=(6, 7))
ax = sns.barplot(data=df, x='Condition', y='Density', hue='Phase',
                 palette={"S": "#8DB081", "G1": "#C8DBC8", "G2M": "#3E4A3B"},
                 hue_order=['G1', 'S', 'G2M'])

for bar in ax.patches:
    height = bar.get_height()
    if height > 0:
        ax.text(bar.get_x() + bar.get_width()/2, height + 0.01,
                f'{height:.2f}', ha='center', fontsize=fontsize-2)

table = np.array([c.values, pc.values])
chi2, p, _, _ = chi2_contingency(table)
js = jensenshannon(c_norm.values, pc_norm.values)

y_max = df['Density'].max()
y_line = y_max * 1.1
ax.plot([0, 0, 1, 1], [y_line-0.01, y_line, y_line, y_line-0.01], lw=1, color='black')
p_text = f"χ²={chi2:.2f}, p={p:.1e}, JS={js:.2f}"
ax.text(0.5, y_line + 0.02, p_text, ha='center', fontsize=fontsize-2)

ax.set_title('Phase Density', fontsize=fontsize)
ax.set_ylabel('Density', fontsize=fontsize)
ax.set_xlabel('')
plt.xticks(fontsize=fontsize)
plt.yticks(fontsize=fontsize)
plt.legend(title='', fontsize=fontsize-2, loc='upper center',
           bbox_to_anchor=(0.5, -0.1), fancybox=True, shadow=False, ncol=3)
plt.gca().set_axisbelow(True)
plt.ylim((0, 0.6))
plt.tight_layout()
plt.grid(linestyle='dotted', axis='y')
plt.savefig('systema_reproduced_big.pdf', bbox_inches='tight')
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 假设 adata_cc 是你打完分的矩阵
# 提取 Control 组细胞
ad_control = adata_cc[adata_cc.obs['control'] == True].copy()

plt.figure(figsize=(6, 6))
sns.scatterplot(
    x=ad_control.obs['S_score'], 
    y=ad_control.obs['G2M_score'],
    hue=ad_control.obs['phase'],
    palette={'G1': '#C8DBC8', 'S': '#8DB081', 'G2M': '#3E4A3B'},
    s=10, alpha=0.6, edgecolor=None
)

# 画两条零线
plt.axhline(0, color='black', linestyle='--', linewidth=1)
plt.axvline(0, color='black', linestyle='--', linewidth=1)

plt.title("Cell Cycle Scores (Control Cells)")
plt.xlabel("S Score")
plt.ylabel("G2M Score")
plt.show()